In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
CACHE_PATH = PROJECT_ROOT / "cache"
CACHE_PATH.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_ROOT))

# import pyfredapi as pf
import pandas as pd
# from fred_api_key import FRED_API_KEY
# from time import sleep

# API_KEY = FRED_API_KEY

START_DATE = "2000-01-01"
END_DATE = "2026-06-30"

In [2]:
from pandas.tseries.holiday import USFederalHolidayCalendar
import pandas_market_calendars as mcal

# Federal holidays (for on_holiday)
cal = USFederalHolidayCalendar()

federal_holidays = cal.holidays(
    start=START_DATE,
    end=END_DATE
)

# NYSE closure holidays (for pre/post effects)
nyse = mcal.get_calendar("NYSE")

# Get actual NYSE trading days in the date range
nyse_schedule = nyse.schedule(
    start_date=START_DATE,
    end_date=END_DATE
)

nyse_trading_days = pd.DatetimeIndex(nyse_schedule.index)

# Find calendar days when NYSE was closed
all_days = pd.date_range(
    start=START_DATE,
    end=END_DATE,
    freq="D"
)

nyse_holidays = all_days.difference(nyse_trading_days)

# Optional: keep only dates (remove time component if present)
nyse_holidays = pd.DatetimeIndex(nyse_holidays.normalize())

In [3]:
import yfinance as yf
# df_stock: indexed by trading dates
df_stock = yf.download(["^GSPC", "^DJI", "^IXIC", "^NDX", "^NYA", "^RUT",
                        # "DX-Y.NYB", "^FTSE", "^N225", "^GDAXI", "^STI", "^TWII", "000001.SS",
                        # "^FCHI", "^STOXX50E", "^GSPTSE", "^BVSP"
                        ],
                        start=START_DATE, end=END_DATE, auto_adjust=True)

df_ref = yf.download(["^VIX", "^VVIX", "DX-Y.NYB", "^FTSE", "^N225", "^GDAXI", "^STI",
                      "^TWII", "000001.SS", "^FCHI", "^STOXX50E", "^GSPTSE", "^BVSP",
                      ],
                      start=START_DATE, end=END_DATE, auto_adjust=True)

df_stock.join(df_ref, how="left")

# holiday_dates: DatetimeIndex from USFederalHolidayCalendar

df_stock["on_holiday"] = 0
df_stock["pre_holiday"] = 0
df_stock["post_holiday"] = 0

# Holidays that occur on trading days
df_stock.loc[df_stock.index.isin(federal_holidays), "on_holiday"] = 1

for h in federal_holidays:
    # Last trading day before holiday
    prev = df_stock.index[df_stock.index < h]
    if len(prev):
        df_stock.loc[prev[-1], "pre_holiday"] = 1

    # First trading day after holiday
    nxt = df_stock.index[df_stock.index > h]
    if len(nxt):
        df_stock.loc[nxt[0], "post_holiday"] = 1

[*********************100%***********************]  6 of 6 completed
[*********************100%***********************]  13 of 13 completed


In [4]:
# print(df_stock.head())
df_stock.to_csv(CACHE_PATH / "indices_from_2000.csv")